In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import config
import photometry_processor

In [ ]:
# --- Quick sanity check: video frame count vs tracking rows ---
# Paste this cell into your notebook and set the paths/column names below.

from pathlib import Path
import cv2
import pandas as pd
import numpy as np

# === CONFIG ===
BEHAVIOR_VIDEO_PATH = Path("/mnt/hdd/Jonathan_IFxDN/AllTopcams/SplitWithGPU/Arena1234_CenterTopCamVid_2024-12-13T13_37_45_top_left.avi")
TRACKING_CSV_PATH   = Path("/mnt/neuropharm/SUN-IN-Neuropharm/Common/Lab member folder/Jonathan Korn/IFxDN Photometry Cohort/OFT/IFxDN_OFTs/T1_1658.740.1666.e/CenterTopCam_AllTimestampsCombined2024-12-13T12_20_06.csv")   # or your tracking file
# If your tracking CSV has a per-frame index column (e.g., "frame", "FrameID", etc.), set it here;
# otherwise leave as None to just count rows.
TRACKING_INDEX_COL = 'FrameCounter'

# If your CSV has a preferred camera timestamp column, set it here (optional).
# Examples from your data: "Cam_Basler_Timestamp" (ns), "FP3002_System_Timestamp" (s)
TIMESTAMP_COL_PREFERRED = "FP3002_System_Timestamp"  # or "FP3002_System_Timestamp" or None

# Whether to do an exact frame count by iterating the file (slower but trustworthy).
ACCURATE_VIDEO_COUNT = False


def count_video_frames(path: Path, accurate: bool = False) -> tuple[int, float]:
    """Return (frame_count, fps). If accurate=True, counts by reading frames."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print(f"[ERROR] Could not open video: {path}")
        return None, None
    fps = cap.get(cv2.CAP_PROP_FPS) or None

    if accurate:
        total = 0
        while True:
            ok, _ = cap.read()
            if not ok:
                break
            total += 1
        cap.release()
        return total, fps

    # Header-reported count (can be wrong for some codecs)
    reported = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    return (int(reported) if reported and reported > 0 else None), fps


def pick_timestamp_column(df: pd.DataFrame, preferred: str = None) -> str:
    """Choose a timestamp column; prefer 'preferred' if present, else try known names."""
    if preferred and preferred in df.columns:
        return preferred
    candidates = [
        "Cam_Basler_Timestamp", "Cam_Bonsai_Timestamp",
        "FP3002_System_Timestamp", "FP3002_Bonsai_Timestamp",
        "timestamp", "time", "Time", "frame_time"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def seconds_scale_from_values(values: np.ndarray) -> float:
    """Infer scale to seconds from magnitude."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return 1.0
    med = float(np.median(np.abs(v)))
    # Heuristics:
    # ~1e12: nanoseconds → divide by 1e9
    # ~1e9:  microseconds → divide by 1e6   (rare here)
    # ~1e6:  milliseconds → divide by 1e3   (rare here)
    # ~1e3–1e5: seconds     → divide by 1
    # ~4.4e4: Excel/Matlab days → (days * 86400)
    if med > 1e10:   # ns
        return 1e9
    if 1e7 < med <= 1e10:  # µs-ish
        return 1e6
    if 1e4 < med <= 1e7:   # ms-ish
        return 1e3
    # Detect days (Excel/Matlab datenum ~ 40000–50000)
    if 3e4 < med < 8e4:
        return 1/86400.0  # multiply by 86400 later (i.e., seconds = value / scale -> value * 86400)
    return 1.0  # assume already seconds


# --- Load tracking CSV ---
try:
    df_trk = pd.read_csv(TRACKING_CSV_PATH)
except Exception as e:
    raise RuntimeError(f"Failed to read tracking CSV at {TRACKING_CSV_PATH}: {e}")

n_rows_total = len(df_trk)
n_rows_effective = n_rows_total
if TRACKING_INDEX_COL and TRACKING_INDEX_COL in df_trk.columns:
    n_rows_effective = df_trk[TRACKING_INDEX_COL].nunique()

ts_col = pick_timestamp_column(df_trk, TIMESTAMP_COL_PREFERRED)
duration_sec = None
implied_fps_from_ts = None
if ts_col:
    vals = pd.to_numeric(df_trk[ts_col], errors="coerce").to_numpy()
    vals = vals[np.isfinite(vals)]
    if vals.size >= 2:
        scale = seconds_scale_from_values(vals)  # if returned 1e9: values are ns; seconds = diff/1e9
        if scale == 1/86400.0:
            # days → seconds
            duration_sec = float(np.nanmax(vals) - np.nanmin(vals)) * 86400.0
        else:
            duration_sec = float(np.nanmax(vals) - np.nanmin(vals)) / scale
        if duration_sec > 0:
            implied_fps_from_ts = n_rows_effective / duration_sec

# --- Video info ---
video_frames, video_fps = count_video_frames(BEHAVIOR_VIDEO_PATH, accurate=ACCURATE_VIDEO_COUNT)
video_duration = (video_frames / video_fps) if (video_frames and video_fps and video_fps > 0) else None

# --- Report ---
print("=== Behavior Video vs Tracking CSV ===")
print(f"Behavior video: {BEHAVIOR_VIDEO_PATH.name}")
print(f"  - FPS (reported): {video_fps if video_fps else 'n/a'}")
print(f"  - Frames ({'accurate' if ACCURATE_VIDEO_COUNT else 'reported'}): {video_frames if video_frames is not None else 'n/a'}")
print(f"  - Duration (from header): {f'{video_duration:.2f} s' if video_duration else 'n/a'}")

print(f"\nTracking CSV: {TRACKING_CSV_PATH.name}")
print(f"  - Rows (total): {n_rows_total}")
if TRACKING_INDEX_COL:
    print(f"  - Unique '{TRACKING_INDEX_COL}': {n_rows_effective}")
else:
    print(f"  - Effective rows (used): {n_rows_effective}  (set TRACKING_INDEX_COL to count uniques if needed)")

if ts_col and duration_sec is not None:
    print(f"  - Timestamp column: {ts_col}")
    print(f"  - Duration (from timestamps): {duration_sec:.2f} s")
    if implied_fps_from_ts:
        print(f"  - Implied FPS (rows/duration): {implied_fps_from_ts:.3f}")

# --- Differences ---
if video_frames is not None:
    diff = video_frames - n_rows_effective
    pct = (diff / video_frames) * 100 if video_frames else np.nan
    print("\n=== Counts Mismatch ===")
    print(f"Video frames - Tracking rows = {diff}  ({pct:+.2f} % of video frames)")
else:
    print("\n[WARN] Couldn't determine video frame count; mismatch calc skipped.")

# Optional drift hint if both durations known
if video_duration and duration_sec and video_duration > 0 and duration_sec > 0:
    fps_video = video_fps if video_fps else (video_frames / video_duration)
    fps_track = implied_fps_from_ts if implied_fps_from_ts else (n_rows_effective / duration_sec)
    if fps_video and fps_track:
        drift_per_min = abs(fps_video - fps_track) * 60.0 / max(fps_video, fps_track)
        print(f"\nDrift indicator: {(fps_video - fps_track):+.4f} FPS difference "
              f"(~{drift_per_min:.2f} frames per minute relative drift)")


RuntimeError: Failed to read tracking CSV at mnt/neuropharm/SUN-IN-Neuropharm/Common/Lab member folder/Jonathan Korn/IFxDN Photometry Cohort/OFT/IFxDN_OFTs/T1_1658.740.1666.e/CenterTopCam_AllTimestampsCombined2024-12-13T12_20_06.csv: [Errno 2] No such file or directory: 'mnt/neuropharm/SUN-IN-Neuropharm/Common/Lab member folder/Jonathan Korn/IFxDN Photometry Cohort/OFT/IFxDN_OFTs/T1_1658.740.1666.e/CenterTopCam_AllTimestampsCombined2024-12-13T12_20_06.csv'

In [2]:
# ---------- robust unit detection ----------
def _divisor_to_seconds(ts):
    ts = np.asarray(ts, dtype=float)
    ts = ts[np.isfinite(ts)]
    if ts.size < 3:
        return 1.0
    diffs = np.diff(ts)
    m = float(np.median(np.abs(diffs)))

    # explicit bands for a ~60 Hz stream
    if 5e6 <= m <= 5e7:   # ~10–50 million → ns per frame
        return 1e9
    if 5e3 <= m <= 5e4:   # ~5k–50k       → µs per frame
        return 1e6
    if 5.0 <= m <= 50.0:  # ~5–50         → ms per frame
        return 1e3
    if 0.001 <= m <= 0.1: # ~0.001–0.1    → s per frame (≈0.0167)
        return 1.0

    # fallback to nearest canonical
    candidates = [(1e9, 1/60 * 1e9), (1e6, 1/60 * 1e6), (1e3, 1/60 * 1e3), (1.0, 1/60)]
    return min(candidates, key=lambda x: abs(m - x[1]))[0]

# ---------- single trial/arena audit (v2) ----------
def _audit_one_v2(center_ts_csv, photometry_csv, arena, fps=60.0, sample=800):
    df_ts = pd.read_csv(center_ts_csv, nrows=None)
    ref_cols = [c for c in ("Cam_Basler_Timestamp", "Cam_Bonsai_Timestamp") if c in df_ts.columns]
    if not ref_cols:
        return [{"status":"error", "msg":"no camera timestamp column found"}]

    # merged photometry once (gives BehaviorTimestamp on camera clock)
    L, R = config.ARENA_TO_PHOTOMETRY_COLS[arena]
    merged = photometry_processor.load_raw_photometry_and_merge_timestamps(
        session_photometry_csv=photometry_csv,
        session_timestamp_csv=center_ts_csv,
        arena=arena, left_region_name=L, right_region_name=R,
        log_prefix=f"[{Path(center_ts_csv).parent.name}|{arena}] "
    )
    if merged is None or merged.empty or "BehaviorTimestamp" not in merged.columns:
        return [{"status":"error", "msg":"failed to get merged photometry"}]

    phot_div = _divisor_to_seconds(merged["BehaviorTimestamp"].to_numpy())
    t0_phot  = float(merged["BehaviorTimestamp"].iloc[0] / phot_div)

    results = []
    for cam_col in ref_cols:
        raw = pd.to_numeric(df_ts[cam_col], errors="coerce").to_numpy()
        raw = raw[np.isfinite(raw)]
        if raw.size == 0:
            results.append({"status":"error", "ref":cam_col, "msg":"camera ts empty"}); continue

        cam_div = _divisor_to_seconds(raw)
        cam_ts  = raw / cam_div

        # sanity: actual camera fps
        med_dt = float(np.median(np.diff(cam_ts)))
        cam_fps = 1.0 / med_dt if med_dt > 0 else np.nan

        n_total = len(cam_ts)
        n_samp  = min(sample, n_total)
        idxs    = (np.linspace(0, n_total-1, n_samp)).astype(int)

        # synthetic vs actual (synthetic is what your plotting code uses)
        t_rel = idxs / float(fps)      # i / 60
        t_syn = t0_phot + t_rel
        t_cam = cam_ts[idxs]

        delta = t_syn - t_cam          # seconds

        mean_ms = float(np.nanmean(delta)*1e3)
        std_ms  = float(np.nanstd(delta)*1e3)

        A = np.vstack([t_rel, np.ones_like(t_rel)]).T
        slope, intercept = np.linalg.lstsq(A, delta, rcond=None)[0]  # sec/sec, sec
        slope_ms_per_s = float(slope*1e3)
        offset_fit_ms  = float(intercept*1e3)

        half = 0.5/fps; one = 1.0/fps
        within_half = float((np.abs(delta) <= half).mean())
        within_one  = float((np.abs(delta) <= one ).mean())

        offset0_ms = float((t0_phot - cam_ts[0])*1e3)

        results.append(dict(
            status="ok", ref=cam_col,
            mean_ms=mean_ms, std_ms=std_ms,
            slope_ms_per_s=slope_ms_per_s, offset_fit_ms=offset_fit_ms,
            offset_at_start_ms=offset0_ms,
            within_half_frame=within_half, within_one_frame=within_one,
            n_samples=int(n_samp),
            cam_fps=cam_fps, med_dt_sec=med_dt,
            used_div_camera=cam_div, used_div_phot=phot_div
        ))
    return results

# ---------- batch audit over all trials (v2) ----------
def audit_all_trials_v2(base_dir, fps=60.0, sample=800, save_csv=True):
    rows = []
    base = Path(base_dir)
    candidates = [p for p in sorted(base.iterdir()) if p.is_dir()]
    for trial_dir in candidates:
        center_csv = next(trial_dir.glob("CenterTopCam_AllTimestampsCombined*.csv"), None)
        phot_csv   = next(trial_dir.glob("photometry_data_combined*.csv"), None)
        if not center_csv or not phot_csv:
            rows.append(dict(trial=trial_dir.name, arena=None, ref=None, status="skip",
                             msg="missing one or both csvs")); 
            continue

        for arena in ("arena1","arena2","arena3","arena4"):
            if arena not in config.ARENA_TO_PHOTOMETRY_COLS:
                rows.append(dict(trial=trial_dir.name, arena=arena, ref=None, status="skip",
                                 msg="arena not in mapping")); 
                continue

            res_list = _audit_one_v2(center_csv, phot_csv, arena, fps=fps, sample=sample)
            for res in res_list:
                row = dict(trial=trial_dir.name, arena=arena)
                row.update(res)
                rows.append(row)

    df = pd.DataFrame(rows)
    if save_csv:
        out = Path(base_dir) / "audit_sync_summary_v2.csv"
        df.to_csv(out, index=False)
        print(f"Saved summary → {out}")

    if not df.empty and "cam_fps" in df.columns:
        ok = df[df.status=="ok"]
        if not ok.empty:
            grp = ok.groupby(["trial","arena","ref"]).agg(
                cam_fps=("cam_fps","mean"),
                mean_ms=("mean_ms","mean"),
                std_ms=("std_ms","mean"),
                drift_ms_per_s=("slope_ms_per_s","mean"),
                offset0_ms=("offset_at_start_ms","mean"),
                within1f=("within_one_frame","mean"),
                used_div_camera=("used_div_camera","first"),
                used_div_phot=("used_div_phot","first"),
            ).reset_index()
            print(grp.head(12))
    return df


In [3]:
BASE = "/mnt/neuropharm/SUN-IN-Neuropharm/Common/Lab member folder/Jonathan Korn/IFxDN Photometry Cohort/OFT/IFxDN_OFTs"
df2 = audit_all_trials_v2(BASE, fps=60.0, sample=800, save_csv=True)


/home/fsp585/Desktop/lundbeck_pres/beautified_code/photometry_processor.py:116: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(iso_valid, phot_valid, 1) # Linear fit
/home/fsp585/Desktop/lundbeck_pres/beautified_code/photometry_processor.py:116: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(iso_valid, phot_valid, 1) # Linear fit
[T1_1658.740.1666.e|arena4] Near-zero std deviation for G14 dF/F. Setting Z-score to 0.
/home/fsp585/Desktop/lundbeck_pres/beautified_code/photometry_processor.py:116: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(iso_valid, phot_valid, 1) # Linear fit
[T2_1664.1678.e.e|arena3] Near-zero std deviation for G8 dF/F. Setting Z-score to 0.
/home/fsp585/Desktop/lundbeck_pres/beautified_code/photometry_processor.py:116: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(iso_valid, phot_valid, 1) # Linear fit
/home/fsp585/Desktop/lundbeck_pres/beautified_code/photometry_processor.py:

Saved summary → /mnt/neuropharm/SUN-IN-Neuropharm/Common/Lab member folder/Jonathan Korn/IFxDN Photometry Cohort/OFT/IFxDN_OFTs/audit_sync_summary_v2.csv
                 trial   arena                   ref    cam_fps       mean_ms  \
0   T1_1658.740.1666.e  arena1  Cam_Basler_Timestamp  80.012264  6.648645e+05   
1   T1_1658.740.1666.e  arena1  Cam_Bonsai_Timestamp  80.046107 -4.249552e+07   
2   T1_1658.740.1666.e  arena2  Cam_Basler_Timestamp  80.012264  6.648645e+05   
3   T1_1658.740.1666.e  arena2  Cam_Bonsai_Timestamp  80.046107 -4.249552e+07   
4   T1_1658.740.1666.e  arena3  Cam_Basler_Timestamp  80.012264  6.648645e+05   
5   T1_1658.740.1666.e  arena3  Cam_Bonsai_Timestamp  80.046107 -4.249552e+07   
6   T1_1658.740.1666.e  arena4  Cam_Basler_Timestamp  80.012264  6.648645e+05   
7   T1_1658.740.1666.e  arena4  Cam_Bonsai_Timestamp  80.046107 -4.249552e+07   
8     T2_1664.1678.e.e  arena1  Cam_Basler_Timestamp  80.012264  6.138814e+05   
9     T2_1664.1678.e.e  arena1  Cam_

In [ ]:
audit_trial(path_example)